In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/creditcard.csv')
df = df.sort_values('Time').reset_index(drop=True)

# Velocity features
df['amount_rolling_10'] = df['Amount'].rolling(10, min_periods=1).mean()
df['amount_rolling_50'] = df['Amount'].rolling(50, min_periods=1).mean()
df['amount_rolling_std_10'] = df['Amount'].rolling(10, min_periods=1).std().fillna(0)

# Amount z-score — how unusual is this transaction vs recent history?
df['amount_zscore'] = (
    (df['Amount'] - df['amount_rolling_10']) / 
    (df['amount_rolling_std_10'] + 1e-8)
)

# Time features
df['hour_of_day'] = (df['Time'] % 86400 // 3600).astype(int)
df['is_night'] = ((df['hour_of_day'] >= 22) | (df['hour_of_day'] <= 5)).astype(int)

# Time since last transaction
df['time_since_last'] = df['Time'].diff().fillna(0)
df['rapid_succession'] = (df['time_since_last'] < 60).astype(int)

print("New features added:")
new_features = ['amount_rolling_10', 'amount_rolling_50', 'amount_zscore',
                'hour_of_day', 'is_night', 'time_since_last', 'rapid_succession']
print(df[new_features].describe().round(3))
print(f"\nFraud rate in night transactions: {df[df.is_night==1]['Class'].mean():.4%}")
print(f"Fraud rate in day transactions:  {df[df.is_night==0]['Class'].mean():.4%}")

New features added:
       amount_rolling_10  amount_rolling_50  amount_zscore  hour_of_day  \
count         284807.000         284807.000     284807.000   284807.000   
mean              88.351             88.354          0.001       14.046   
std               81.649             40.505          0.950        5.836   
min                0.010              1.581         -2.846        0.000   
25%               40.414             61.436         -0.573       10.000   
50%               66.474             81.879         -0.376       15.000   
75%              109.727            107.139          0.180       19.000   
max             2712.061            602.899          2.846       23.000   

         is_night  time_since_last  rapid_succession  
count  284807.000       284807.000          284807.0  
mean        0.177            0.607               1.0  
std         0.381            1.053               0.0  
min         0.000            0.000               1.0  
25%         0.000            

In [2]:
# Fix rapid_succession — use 10 seconds instead of 60
df['rapid_succession'] = (df['time_since_last'] < 10).astype(int)

print(f"Rapid succession rate: {df['rapid_succession'].mean():.4%}")
print(f"Fraud rate in rapid transactions: {df[df.rapid_succession==1]['Class'].mean():.4%}")
print(f"Fraud rate in normal transactions: {df[df.rapid_succession==0]['Class'].mean():.4%}")

Rapid succession rate: 99.8276%
Fraud rate in rapid transactions: 0.1716%
Fraud rate in normal transactions: 0.8147%


In [3]:
from sklearn.preprocessing import StandardScaler

# Drop rapid_succession — not meaningful for this dataset
df = df.drop(columns=['rapid_succession', 'time_since_last'])

# Scale Amount and Time
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

# Final feature check
print(f"Final dataset shape: {df.shape}")
print(f"\nNew features added:")
print([c for c in df.columns if c not in 
       [f'V{i}' for i in range(1,29)] + ['Time','Amount','Class']])

# Save
df.to_csv('../data/processed/creditcard_features.csv', index=False)
print(f"\n✅ Saved to data/processed/creditcard_features.csv")

Final dataset shape: (284807, 39)

New features added:
['amount_rolling_10', 'amount_rolling_50', 'amount_rolling_std_10', 'amount_zscore', 'hour_of_day', 'is_night', 'Amount_scaled', 'Time_scaled']

✅ Saved to data/processed/creditcard_features.csv
